# Notebook 3 — Vector (Semantic) Search

*Hands-on time: ~35 minutes*

This is the core notebook where we leverage the **dense_vector** embeddings
we indexed in Notebook 1. You'll learn:

1. Basic kNN search with natural language
2. Filtered kNN (restrict by metadata)
3. Hybrid search (keyword + vector combined)
4. Comparing search strategies side-by-side
5. Tuning `num_candidates` and similarity thresholds

**Prerequisites:** Notebooks 1 and 2 completed.

In [ ]:
from elasticsearch import Elasticsearch
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

client = Elasticsearch("http://localhost:9200")
model = SentenceTransformer("all-MiniLM-L6-v2")
INDEX_NAME = "neuroimaging"

print(f"Documents: {client.count(index=INDEX_NAME)['count']}")
print(f"Model dims: {model.get_sentence_embedding_dimension()}")


def embed_query(text: str) -> list:
    """Encode a text query to a vector."""
    return model.encode(text).tolist()


def show_hits(response, fields=None, show_score=True):
    """Pretty-print search results."""
    rows = []
    for hit in response["hits"]["hits"]:
        row = {}
        if show_score:
            row["_score"] = round(hit["_score"], 4) if hit["_score"] else None
        src = hit["_source"]
        if fields:
            for f in fields:
                row[f] = src.get(f)
        else:
            row.update({k: v for k, v in src.items() if k != "metadata_embedding"})
        rows.append(row)
    return pd.DataFrame(rows)

---
## 1. Basic kNN Search

The `knn` parameter performs approximate nearest neighbor search using the
HNSW index on our `metadata_embedding` field.

The key parameters:
- **k**: Number of nearest neighbors to return
- **num_candidates**: How many candidates HNSW considers per shard (bigger = more accurate but slower)

In [ ]:
# Natural language query → vector → kNN search
query_text = "functional brain imaging during a risk-taking task"

response = client.search(
    index=INDEX_NAME,
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": 5,
        "num_candidates": 50
    }
)

print(f'Query: "{query_text}"')
print(f"Hits: {response['hits']['total']['value']}\n")
show_hits(response, ["suffix", "subject", "task", "description_text"])

In [ ]:
# Try another query — notice how semantic understanding works
query_text = "high-resolution brain anatomy scan"

response = client.search(
    index=INDEX_NAME,
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": 5,
        "num_candidates": 50
    }
)

print(f'Query: "{query_text}"')
show_hits(response, ["suffix", "subject", "task", "description_text"])

In [ ]:
# Even indirect queries work — the model captures semantic meaning
queries = [
    "MRI scan on a Siemens scanner",
    "neuroimaging with short TR for fast acquisition",
    "structural brain image of a young participant",
]

for q in queries:
    resp = client.search(
        index=INDEX_NAME,
        knn={
            "field": "metadata_embedding",
            "query_vector": embed_query(q),
            "k": 3,
            "num_candidates": 50
        }
    )
    top = resp["hits"]["hits"][0]["_source"]
    score = resp["hits"]["hits"][0]["_score"]
    print(f'Q: "{q}"')
    print(f'  → top hit (score={score:.4f}): {top["suffix"]} | sub-{top["subject"]} | {top.get("description_text", "")[:80]}\n')

---
## 2. Filtered kNN

You can restrict the kNN search to documents matching a filter.
ES applies the filter **during** the HNSW traversal, not after — ensuring
you always get `k` results that match the filter.

In [ ]:
# kNN search restricted to BOLD scans only
query_text = "task-related brain activation"

response = client.search(
    index=INDEX_NAME,
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": 5,
        "num_candidates": 50,
        "filter": {
            "term": {"suffix": "bold"}
        }
    }
)

print(f'Filtered kNN (BOLD only): "{query_text}"')
show_hits(response, ["suffix", "subject", "task", "run", "RepetitionTime"])

In [ ]:
# Filter by subject AND suffix using a bool filter
query_text = "brain imaging experiment"

response = client.search(
    index=INDEX_NAME,
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": 3,
        "num_candidates": 50,
        "filter": {
            "bool": {
                "must": [
                    {"terms": {"subject": ["01", "02", "03"]}},
                    {"term": {"suffix": "bold"}}
                ]
            }
        }
    }
)

print("Filtered kNN (subjects 01-03, BOLD):")
show_hits(response, ["subject", "suffix", "task", "run", "description_text"])

---
## 3. Hybrid Search — Keyword + Vector Combined

Hybrid search uses both the `query` (BM25) and `knn` parameters simultaneously.
ES computes both scores and combines them. You can use `boost` to weight them.

In [ ]:
# Hybrid: BM25 on description_text + kNN on embedding
query_text = "functional MRI during risk decision making"

response = client.search(
    index=INDEX_NAME,
    query={
        "match": {
            "description_text": {
                "query": query_text,
                "boost": 0.3   # weight for BM25 component
            }
        }
    },
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": 5,
        "num_candidates": 50,
        "boost": 0.7   # weight for vector component
    },
    size=5
)

print(f'Hybrid search: "{query_text}"')
print(f"(BM25 boost=0.3, kNN boost=0.7)\n")
show_hits(response, ["suffix", "subject", "task", "description_text"])

In [ ]:
# Hybrid with a filter on the kNN side
query_text = "high-field MRI structural scan"

response = client.search(
    index=INDEX_NAME,
    query={
        "match": {
            "description_text": {
                "query": query_text,
                "boost": 0.5
            }
        }
    },
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": 5,
        "num_candidates": 50,
        "boost": 0.5,
        "filter": {
            "term": {"suffix": "T1w"}
        }
    },
    size=5
)

print(f'Hybrid + filter: "{query_text}" (T1w only in kNN)\n')
show_hits(response, ["suffix", "subject", "description_text"])

---
## 4. Comparing Search Strategies

Let's run the same query through all three strategies and compare results.

In [ ]:
query_text = "brain scan during a gambling task"
K = 5

# Strategy 1: BM25 only
bm25_resp = client.search(
    index=INDEX_NAME,
    query={"match": {"description_text": query_text}},
    size=K
)

# Strategy 2: kNN only
knn_resp = client.search(
    index=INDEX_NAME,
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": K,
        "num_candidates": 50
    }
)

# Strategy 3: Hybrid
hybrid_resp = client.search(
    index=INDEX_NAME,
    query={"match": {"description_text": {"query": query_text, "boost": 0.3}}},
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": K,
        "num_candidates": 50,
        "boost": 0.7
    },
    size=K
)

print(f'Query: "{query_text}"\n')

for name, resp in [("BM25 Only", bm25_resp), ("kNN Only", knn_resp), ("Hybrid", hybrid_resp)]:
    print(f"\n{'='*60}")
    print(f"Strategy: {name} (hits: {resp['hits']['total']['value']})")
    print(f"{'='*60}")
    display(show_hits(resp, ["suffix", "subject", "task", "description_text"]))

### Observations

- **BM25**: Matches literal terms ("brain", "scan", "task"). May miss documents that
  describe the same concept with different words.
- **kNN**: Understands semantic similarity. \"gambling task\" is close to \"risk-taking task\"
  in embedding space, even if those exact words don't appear.
- **Hybrid**: Best of both worlds — exact matches get boosted by BM25, while kNN
  surfaces semantically relevant results that BM25 would miss.

---
## 5. Tuning: num_candidates and Similarity Thresholds

In [ ]:
# Effect of num_candidates on results
# More candidates = more accurate (closer to exact brute-force) but slower
query_vec = embed_query("structural MRI scan")

for nc in [10, 50, 100, 200]:
    resp = client.search(
        index=INDEX_NAME,
        knn={
            "field": "metadata_embedding",
            "query_vector": query_vec,
            "k": 5,
            "num_candidates": nc
        }
    )
    scores = [h["_score"] for h in resp["hits"]["hits"]]
    ids = [h["_id"] for h in resp["hits"]["hits"]]
    print(f"num_candidates={nc:4d}  top_score={scores[0]:.4f}  ids={ids}")

In [ ]:
# Similarity threshold: post-filter results below a score
query_text = "diffusion tensor imaging white matter tracts"

response = client.search(
    index=INDEX_NAME,
    knn={
        "field": "metadata_embedding",
        "query_vector": embed_query(query_text),
        "k": 5,
        "num_candidates": 100,
        "similarity": 0.5  # minimum cosine similarity threshold
    }
)

print(f'Query: "{query_text}" (similarity >= 0.5)')
print(f"Returned: {len(response['hits']['hits'])} hits")
show_hits(response, ["suffix", "subject", "description_text"])

> **Note:** If the similarity threshold filters out all documents, it means
> our dataset (ds001) doesn't contain DWI scans. This is expected — the dataset
> only has T1w anatomical and BOLD functional scans. The query still demonstrates
> that the system correctly reports low similarity for semantically distant content.

In [ ]:
# Exact kNN (brute force) using script_score — useful for comparison / debugging
query_vec = embed_query("functional MRI brain activation")

response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "script_score": {
                "query": {"match_all": {}},
                "script": {
                    "source": "cosineSimilarity(params.query_vector, 'metadata_embedding') + 1.0",
                    "params": {"query_vector": query_vec}
                }
            }
        },
        "size": 5
    }
)

print("Exact kNN via script_score (brute force):")
print("(Score = cosineSimilarity + 1.0, so 1.0 = orthogonal, 2.0 = identical)\n")
show_hits(response, ["suffix", "subject", "task", "description_text"])

---
## Exercise

1. Write a kNN query for "pediatric brain development study" — what does
   the top result look like and why?
2. Create a hybrid search that finds BOLD scans semantically similar to
   "cognitive control experiment" with BM25 boost=0.2 and kNN boost=0.8
3. Experiment with `similarity` thresholds: at what value do you start
   losing relevant results for "anatomical T1 MRI scan"?

In [ ]:
# YOUR CODE HERE

---
## Summary

You've now mastered:
- ✅ Basic kNN search with natural language queries
- ✅ Filtered kNN to restrict semantic search to structured criteria
- ✅ Hybrid search combining BM25 + kNN with boost weighting
- ✅ Side-by-side comparison of search strategies
- ✅ Tuning `num_candidates` for accuracy/speed tradeoff
- ✅ Similarity thresholds and exact kNN via `script_score`

**Next:** Open **Notebook 4** for a guided tour of **Kibana** to visualize and
interact with this data through a GUI.